# பாடம் 01 - AI முகவரிகள் அறிமுகம்

**ஆரம்பக்காரர்களுக்கான AI முகவரிகள்** பாடத்தின் முதல் பாடத்திற்கு வருக!

ஒரு **AI முகவர்** என்பது அதன் தீர்க்கதரிசன இயந்திரமாக ஒரு பெரிய மொழி மாதிரியை (LLM) பயன்படுத்தும் ஒரு நிரல் ஆகும் மற்றும் ஒரு பயனருக்காக ஒரு குறிக்கோளை நிறைவேற்றக் கூடிய *செயற்பாடுகளை* உண்மையான உலகில் எடுக்கலாம் — API களை அழைக்கவும், தரவுத்தளங்களை விசாரிக்கவும், அல்லது குறியீட்டை இயக்கவும்.

இந்த நோட்புக்கில் நீங்கள் உங்கள் முதல் முகவரான ஒரு **பயண முகவர்** ஐ கட்டமைப்பீர்கள், இது விடுமுறை இடங்களை பரிந்துரைக்கிறது. இதையடுத்து நீங்கள் கற்றுக்கொள்ளப்போகிறீர்கள்:

1. **Microsoft Agent Framework** ஐ பயன்படுத்தி Microsoft Foundry Agent சேவைக்குச் இணைபது.
2. முகவருக்கு ஒரு **கருவி** கொடுப்பது — அது அழைக்கக்கூடிய ஒரு எளிய பைத்தான் செயல்பாடு.
3. முகவரைக் இயக்கி அதன் பதிலை ஆய்வு செய்தல்.
4. முகவரின் பதிலை டோக்கன் வாரியாக தாங்குதல்.


## கட்டமைப்பு

இந்த நோட்புக் இயக்குவதற்கு முன்னர், கீழ்கண்டவைகளை உறுதி பண்ணிக் கொள்ளுங்கள்:

1. **ஒன்றைக்கொண்ட Microsoft Foundry திட்டம்** மற்றும் இயக்கப்பட்ட சேட் மாதிரி (உதா. `gpt-5-mini`) இருக்கும்.
2. **Azure CLI மூலம் உள்நுழைந்திருத்தல்** — உங்கள் டெர்மினலில் `az login` ஐ இயக்கவும்.
3. **தேவைப்பட்ட சுற்றுச்சூழல் மாறிகள் அமைக்கப்பட்டுள்ளன:**
   - `AZURE_AI_PROJECT_ENDPOINT` — உங்கள் Microsoft Foundry திட்டத்தின் எண்ட் பாயிண்ட்.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — உங்கள் இயக்கப்பட்ட மாதிரியின் பெயர்.

கீழே உள்ள செல், நீங்கள் தேவையான Python தொகுப்புகளை நிறுவும்.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## உங்கள் முதல் முகவரியை உருவாக்குதல்

ஒரு முகவரிக்கு இரண்டு விஷயங்கள் தேவை:

- **வழிமுறைகள்** அது *யார்* மற்றும் *எப்படி நடக்க வேண்டும்* என்பதை சொல்கின்றன (ஒரு அமைப்பு பணியூக்கம்).
- **கருவிகள்** — `@tool` என அழைக்கப்படும் பைதான் செயல்பாடுகள், அவற்றை முகவரி தகவலைப் பெற அல்லது செயல்களை செய்வதற்காக அழைக்க முடிகிறது.

கீழே நாம் பிரபலமான விடுமுறை இடங்களின் பட்டியலை மீட்டெடுக்கும் ஒரு எளிய கருவியை வரையறுக்கிறோம். பயனர் பயண பரிந்துரைகள் கேட்டால் முகவர் இந்த கருவியை பயன்படுத்தும்.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## ஸ்ட்ரீமிங் பதில்கள்

மேலும் தொடர்புடைய அனுபவத்திற்காக, நீங்கள் முகவர் பதில்களை **ஸ்ட்ரீம்** செய்யலாம். முழு பதிலை காத்திருப்பதையும் மாற்றி, முகவர் உருவாக்கப்படும் போது உரை துண்டுகளை வழங்குகிறது. உண்மையான நேரத்தில் வெளியீட்டை காட்ட விரும்பும் சந்தைப் பரிமாற்ற இடைமுகங்களில் இது மிகவும் பயனுள்ளது.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் எப்படி செய்வது என்று கற்றுக்கொண்டீர்கள்:

- **`FoundryChatClient` மூலம் Microsoft Foundry Agent Service-இனை இணைக்கும் ஒரு வழங்குநரை உருவாக்கவும்**.
- **`@tool` அலங்காரத்தைப் பயன்படுத்தி ஒரு கருவியை வரையறுக்கவும், அப்படியே முகவரியால் உங்கள் Python செயல்பாடுகளை அழைக்க முடியும்**.
- **ஒரு பயனர் செய்தியுடன் முகவரியை இயக்கவும் மற்றும் அதன் பதிலை அச்சிடவும்**.
- **உண்மையான நேர வெளியீட்டிற்கான பதில்களை தொடர்ச்சியாக வழங்கவும்**.

அடுத்த பாடத்தில் நாங்கள் முகவரிகள் கட்டமைப்புகளை இன்னும் விரிவாகப் பார்வையிடுவோம் மற்றும் முகவரிகளுக்கு அதிக சக்தி வாய்ந்த கருவிகள் மற்றும் பல படி காரணீக்ய திறன்களை வழங்குவது எப்படி என்று கற்றுக்கொள்வோம்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
